# Cypher L4 — PaddleOCR fringe-case extractor

**When to use this notebook**: a PDF (or specific pages within it) failed to extract via the local Cypher pipeline (L1 / L2 / L3) because of OCR-quality issues — typically bordered tables, dense scans, or Asian-carrier source scans where Tesseract garbles dates and PNs.

**What it does**:
1. Installs PaddleOCR with the PP-Structure table-recognition model.
2. Lets you upload a PDF.
3. Renders each requested page and runs PaddleOCR.
4. Writes the OCR output to CSV so you can merge it back into your local results.

**Why PaddleOCR over Tesseract**: PaddleOCR has a dedicated table-structure model (PP-Structure) that explicitly understands ruled tables. On the kind of OCCM scans where Tesseract reads `[`, `|`, `_` as content and garbles date columns, PaddleOCR typically reconstructs the cell grid.

**Runtime**: GPU-enabled Colab is much faster (~3 s/page vs ~15 s on CPU). `Runtime → Change runtime type → T4 GPU`.

**Output**: `cypher_L4_<filename>.csv`, plus optional XLSX, downloaded automatically. Each row is tagged `_source: L4_paddle` and `_page: N` so you can merge it into your main results CSV by `_page`.

## 1. Install

In [ ]:
%pip install --quiet paddlepaddle paddleocr pdf2image pymupdf pandas openpyxl
!apt-get install -y -qq poppler-utils  # pdf2image dependency

## 2. Upload PDF

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_name = next(iter(uploaded))
print(f'Uploaded: {pdf_name}  ({len(uploaded[pdf_name]):,} bytes)')

## 3. Configure variant + page range

Pick which OCCM variant this is (controls the column schema and validation), and optionally restrict to specific pages where L3 failed. Leave `pages = None` to process the whole document.

In [ ]:
VARIANT = 'China Eastern'   # 'China Eastern' | 'Aeroflot' | 'AMOS'
PAGES   = None              # None for all, or a list like [4, 5, 6, 12, 13, 14]
DPI     = 300               # bump to 400 for low-quality scans

VARIANT_COLUMNS = {
    'China Eastern': ['ATA', 'DESCRIPTION', 'FIN', 'PART_NUMBER', 'SERIAL_NUMBER', 'DATE', 'FH', 'FC'],
    'Aeroflot':      ['ATA', 'ZONE', 'FIN', 'DESCRIPTION', 'VENDOR_CODE', 'PART_NUMBER', 'SERIAL_NUMBER'],
    'AMOS':          ['ATA', 'PART_NUMBER', 'SERIAL_NUMBER', 'DESCRIPTION', 'POS', 'RELEASE_LABEL', 'INST_DATE', 'TSN', 'CSN'],
}
columns = VARIANT_COLUMNS[VARIANT]
print(f'Variant: {VARIANT}\nColumns: {columns}\nPages:   {PAGES or "all"}\nDPI:     {DPI}')

## 4. Render pages and run PaddleOCR

We use PP-Structure with the table model. For each page we get a structured representation of the table cells, then map those cells onto the variant's column schema.

In [ ]:
import fitz  # pymupdf
from PIL import Image
import io, numpy as np
from paddleocr import PPStructure

# Initialize the structure model — reuses cached weights between runs
structure_engine = PPStructure(table=True, ocr=True, show_log=False, lang='en')

doc = fitz.open(pdf_name)
page_indices = [i - 1 for i in PAGES] if PAGES else list(range(len(doc)))
print(f'Will process {len(page_indices)} pages.')

In [ ]:
all_records = []
for page_idx in page_indices:
    page = doc[page_idx]
    pix = page.get_pixmap(dpi=DPI)
    img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
    arr = np.array(img)
    print(f'\nPage {page_idx + 1}: {arr.shape[1]}x{arr.shape[0]} px')
    result = structure_engine(arr)
    # Each `result` is a list of regions (text/title/table). Pull tables.
    page_rows = []
    for region in result:
        if region.get('type') != 'table':
            continue
        # PP-Structure returns `res['html']` for tables; we want raw cells.
        cells = region.get('res', {}).get('cells')  # may be None depending on version
        # Fall back to parsing HTML rows
        html = region.get('res', {}).get('html', '')
        if html:
            import re
            tr_blocks = re.findall(r'<tr.*?>(.*?)</tr>', html, re.DOTALL)
            for tr in tr_blocks:
                tds = re.findall(r'<td.*?>(.*?)</td>', tr, re.DOTALL)
                row = [re.sub(r'<[^>]+>', '', td).strip() for td in tds]
                if not row or not row[0]:
                    continue
                # Map first N cells onto the column schema
                rec = {col: (row[i] if i < len(row) else '') for i, col in enumerate(columns)}
                rec['_page'] = page_idx + 1
                rec['_source'] = 'L4_paddle'
                page_rows.append(rec)
    all_records.extend(page_rows)
    print(f'  → {len(page_rows)} rows extracted')

print(f'\nTotal L4 rows extracted: {len(all_records)}')

## 5. Inspect, save, download

The output is **raw OCR**, not yet validated against Cypher's per-cell rules. Eyeball the first 20 rows; if they look plausible, save and download. Merging into the main local results is a one-liner: `pd.concat([main_csv, l4_csv]).sort_values(['_page'])`.

In [ ]:
import pandas as pd
df = pd.DataFrame(all_records)
df.head(20)

In [ ]:
import pathlib
stem = pathlib.Path(pdf_name).stem
out_csv  = f'cypher_L4_{stem}.csv'
out_xlsx = f'cypher_L4_{stem}.xlsx'
df.to_csv(out_csv, index=False)
df.to_excel(out_xlsx, index=False)
files.download(out_csv)
files.download(out_xlsx)
print(f'Downloaded {out_csv} and {out_xlsx}')

## 6. Merge into local results (do this on your machine)

Move the downloaded CSV into `cypher/research/results/by_pdf/`, then run a small merge script to combine it with the existing `_L1.csv` for the same PDF. The combined output has rows from both sources, distinguishable by `_source`. Then re-run `python research/report_builder.py` to refresh the dashboard.